# 08 CNN follow-up: SpecAugment and a lower learning rate for architecture C (validation first, test once)

Notebook 06 compared three depths at one learning rate with no augmentation. Here the selected architecture (C: 4 blocks + dropout) is trained three ways in one session: unchanged, with SpecAugment, and at a lower learning rate. The variant with the best validation macro-F1 is scored once on test. Runs on Kaggle like notebook 06 (GPU T4, about 30 minutes).

**On Kaggle:** add the competition data as an input, set the accelerator to GPU T4 x2 and turn Internet on. The next cell clones the repo, points `SPEC_DIR` at Kaggle's copy of the data and rebuilds the split index; it does nothing when run anywhere else.

In [ ]:
import os, subprocess, sys
from pathlib import Path
if Path("/kaggle/input").exists():
    import pandas as pd
    repo = Path("/tmp/F21DL")  # outside /kaggle/working, so the clone (and any token) never lands in the saved output
    if not repo.exists():
        url = "github.com/MrNoobri/F21DL.git"
        try:  # private repo: a read-only GitHub token stored as the Kaggle secret GITHUB_TOKEN
            from kaggle_secrets import UserSecretsClient
            url = f"{UserSecretsClient().get_secret('GITHUB_TOKEN')}@{url}"
        except Exception:
            pass  # public repo: no token needed
        if subprocess.run(["git", "clone", "--depth", "1", f"https://{url}", str(repo)], capture_output=True).returncode:
            raise RuntimeError("git clone failed: turn Internet on, and add the GITHUB_TOKEN secret if the repo is private")
    os.environ["SPEC_DIR"] = str(next(Path("/kaggle/input").rglob("spectrograms_unnormalized")).parent)
    os.chdir(repo / "notebooks")
    committed = pd.read_csv("../data/splits/spec_splits.csv", dtype=str)
    subprocess.run([sys.executable, "../scripts/prepare_data.py", "spec"], check=True)
    assert pd.read_csv("../data/splits/spec_splits.csv", dtype=str).equals(committed), "regenerated splits differ from the committed ones"
    print("Kaggle setup done, data at", os.environ["SPEC_DIR"])

In [ ]:
import sys; sys.path.append('..')
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder
from src.config import FIGURES, RESULTS, SEED
from src.data import load_spec_index, load_spectrograms
from src.evaluate import evaluate, plot_confusion, results_table, save_result

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus or "none, training on CPU")
if gpus:
    # float16 maths (faster on tensor cores, half the memory) with float32 master weights.
    keras.mixed_precision.set_global_policy("mixed_float16")

## 1. Load and normalise (same as notebook 06)

In [ ]:
idx = load_spec_index()
le = LabelEncoder().fit(idx["genre"])  # alphabetical, so softmax column i is le.classes_[i]

def load(split):
    part = idx[idx["split"] == split]
    X = np.empty((len(part), 128, 431, 1), "float32")  # preallocated, so batches never need a second full copy
    for s in range(0, len(part), 1000):
        X[s:s + 1000, ..., 0] = load_spectrograms(part["path"].iloc[s:s + 1000])
    return X, le.transform(part["genre"]), part

(Xtr, ytr, _), (Xva, yva, _), (Xte, yte, test_part) = load("training"), load("validation"), load("test")

# Per-band standardisation with training statistics only (decided in notebook 05):
# each mel band gets mean 0 and std 1 over all training clips and frames.
mean = Xtr.mean(axis=(0, 2, 3), dtype="float64")[:, None, None].astype("float32")
std = Xtr.std(axis=(0, 2, 3), dtype="float64")[:, None, None].astype("float32")
for X in (Xtr, Xva, Xte):
    X -= mean  # in place, to avoid another 2.7 GB copy
    X /= std
Xtr.shape, Xva.shape, Xte.shape

## 2. Three training set-ups for architecture C

**SpecAugment** (Park et al., 2019) sets one random band of up to 16 mel bins and one random stretch of up to 40 frames to 0 in each training clip, which after per-band standardisation is the band's average. The model cannot rely on any single band or moment, which should reduce overfitting. Masks are drawn fresh every batch and are off for validation and test. The other settings match notebook 06: batch 32, up to 40 epochs, EarlyStopping (patience 6, best weights restored) and ReduceLROnPlateau (halve after 3).

In [ ]:
class SpecAugment(keras.layers.Layer):
    """Zeroes one random frequency band and one random time stretch per clip, during training only."""
    def __init__(self, max_bins=16, max_frames=40, **kwargs):
        kwargs.setdefault("dtype", "float32")  # masking needs no float16 maths; the next Conv2D casts as usual
        super().__init__(**kwargs)
        self.max_bins, self.max_frames = max_bins, max_frames

    def mask(self, n, size, max_width):
        # True inside a band [start, start + width) of random width 0..max_width, one band per clip: shape (n, size).
        width = tf.random.uniform((n, 1), 0, max_width + 1, dtype=tf.int32)
        start = tf.random.uniform((n, 1), 0, size - max_width, dtype=tf.int32)
        pos = tf.range(size)[None, :]
        return (pos >= start) & (pos < start + width)

    def call(self, x, training=None):
        if not training:
            return x
        n = tf.shape(x)[0]
        f = self.mask(n, x.shape[1], self.max_bins)[:, :, None, None]
        t = self.mask(n, x.shape[2], self.max_frames)[:, None, :, None]
        return tf.where(f | t, tf.zeros_like(x), x)

EPOCHS = 40
C = dict(filters=(32, 64, 128, 256), dropout=0.3)
setups = {"C (as in 06)": dict(lr=1e-3, augment=False),
          "C + SpecAugment": dict(lr=1e-3, augment=True),
          "C, lr 1e-4": dict(lr=1e-4, augment=False)}

def build(filters, dropout, lr, augment):
    tf.keras.utils.set_random_seed(SEED)
    x = inp = keras.Input((128, 431, 1))
    if augment:
        x = SpecAugment()(x)
    for f in filters:
        x = keras.layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.ReLU()(x)
        x = keras.layers.MaxPooling2D(2)(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(dropout)(x)
    out = keras.layers.Dense(len(le.classes_), activation="softmax", dtype="float32")(x)
    model = keras.Model(inp, out)
    model.compile(keras.optimizers.Adam(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

# Quick check that the masks work: training=True changes some values, training=False changes none.
demo = Xtr[:4]
assert np.array_equal(SpecAugment()(demo, training=False).numpy(), demo)
print("share of values masked in a demo batch:", float(np.mean(SpecAugment()(demo, training=True).numpy() != demo)))

In [ ]:
runs = {}
for name, s in setups.items():
    print(f"\n=== {name} ===")
    model = build(**C, **s)
    t = time.perf_counter()
    hist = model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=EPOCHS, batch_size=32, verbose=2,
                     callbacks=[keras.callbacks.EarlyStopping("val_loss", patience=6, restore_best_weights=True),
                                keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.5, patience=3)])
    epochs = len(hist.history["loss"])
    proba = model.predict(Xva, batch_size=64, verbose=0)
    m, _ = evaluate(le.classes_[yva], le.classes_[proba.argmax(1)], proba, labels=le.classes_)
    runs[name] = {"model": model, "history": hist.history, "epochs": epochs,
                  "best_epoch": int(np.argmin(hist.history["val_loss"])) + 1,
                  "s_per_epoch": round((time.perf_counter() - t) / epochs, 1),
                  "best_val_loss": round(min(hist.history["val_loss"]), 4),
                  "final_train_acc": round(hist.history["accuracy"][-1], 4),
                  "val_accuracy": m["accuracy"], "val_macro_f1": m["macro_f1"], "val_mcc": m["mcc"]}

## 3. Comparison (validation only)

In [ ]:
table = pd.DataFrame({k: {c: v for c, v in r.items() if c not in ("model", "history")} for k, r in runs.items()}).T
selected = table["val_macro_f1"].astype(float).idxmax()
print("Selected on validation macro-F1:", selected)
table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, r in runs.items():
    axes[0].plot(r["history"]["val_loss"], label=name)
    axes[1].plot(r["history"]["accuracy"], ls="--", alpha=0.6)
    axes[1].plot(r["history"]["val_accuracy"], label=name, color=axes[1].lines[-1].get_color())
axes[0].set(xlabel="Epoch", ylabel="Validation loss", title="Validation loss")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Accuracy (dashed: training, solid: validation)")
axes[0].legend()
fig.tight_layout()
fig.savefig(FIGURES / "spec_cnn_followup_curves.png", dpi=200)

## 4. Test (once) for the selected set-up

In [ ]:
best = runs[selected]
proba = best["model"].predict(Xte, batch_size=64, verbose=0)
metrics, per_class = evaluate(le.classes_[yte], le.classes_[proba.argmax(1)], proba, labels=le.classes_)
save_result("cnn_followup", "spec", metrics, params={"setup": selected, **setups[selected], "epochs": best["epochs"]})
results_table("spec")[["accuracy", "macro_f1", "mcc", "macro_auc_ovr"]]

In [ ]:
per_class.sort_values("recall_tpr")

In [ ]:
# On Kaggle: bundle this run's figure and result for download (Output tab: cnn_followup_outputs.zip). Extract it into the repo root.
if Path("/kaggle/working").exists():
    import zipfile
    with zipfile.ZipFile("/kaggle/working/cnn_followup_outputs.zip", "w") as z:
        for f in [FIGURES / "spec_cnn_followup_curves.png", RESULTS / "spec" / "cnn_followup.json"]:
            z.write(f, f.relative_to(FIGURES.parent))
    print(z.namelist())

## Findings

- FILL after the Kaggle run